<a href="https://colab.research.google.com/github/mdimssptr/UPRAK_G211220104-FUZZYLOGIC/blob/main/G211220104_UPRAKFUZZYLOGICDBD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np

# =====================================================
# KASUS A: FUZZY MAMDANI - DIAGNOSA RISIKO DBD
# =====================================================

class FuzzyMamdani:
    def __init__(self):
        self.suhu = None
        self.trombosit = None
        self.leukosit = None

    def fuzzifikasi_suhu(self, suhu):
        """Fuzzifikasi suhu dengan fungsi keanggotaan segitiga"""
        # Normal [36-37.5]
        if suhu <= 36:
            normal = 1
        elif 36 < suhu < 37.5:
            normal = (37.5 - suhu) / (37.5 - 36)
        else:
            normal = 0

        # Demam [37-39]
        if suhu <= 37 or suhu >= 39:
            demam = 0
        elif 37 < suhu <= 38:
            demam = (suhu - 37) / (38 - 37)
        elif 38 < suhu < 39:
            demam = (39 - suhu) / (39 - 38)
        else:
            demam = 1

        # Tinggi [38-41]
        if suhu <= 38:
            tinggi = 0
        elif 38 < suhu <= 39:
            tinggi = (suhu - 38) / (39 - 38)
        elif 39 < suhu < 41:
            tinggi = 1
        else:
            tinggi = 1

        return {'normal': normal, 'demam': demam, 'tinggi': tinggi}

    def fuzzifikasi_trombosit(self, trombosit):
        """Fuzzifikasi trombosit"""
        # Rendah [<150]
        if trombosit <= 100:
            rendah = 1
        elif 100 < trombosit < 150:
            rendah = (150 - trombosit) / (150 - 100)
        else:
            rendah = 0

        # Normal [150-400]
        if trombosit <= 100:
            normal = 0
        elif 100 < trombosit <= 150:
            normal = (trombosit - 100) / (150 - 100)
        elif 150 < trombosit <= 400:
            normal = 1
        else:
            normal = 0

        return {'rendah': rendah, 'normal': normal}

    def fuzzifikasi_leukosit(self, leukosit):
        """Fuzzifikasi leukosit"""
        # Rendah [<4]
        if leukosit <= 3:
            rendah = 1
        elif 3 < leukosit < 4:
            rendah = (4 - leukosit) / (4 - 3)
        else:
            rendah = 0

        # Normal [4-10]
        if leukosit <= 3:
            normal = 0
        elif 3 < leukosit <= 4:
            normal = (leukosit - 3) / (4 - 3)
        elif 4 < leukosit <= 10:
            normal = 1
        else:
            normal = 0

        return {'rendah': rendah, 'normal': normal}

    def inferensi(self, fuzz_suhu, fuzz_trombosit, fuzz_leukosit):
        """Evaluasi rules dengan operator MIN"""
        rules = []

        # R1: Tinggi ∧ T.Rendah ∧ L.Rendah → Tinggi
        alpha_r1 = min(fuzz_suhu['tinggi'], fuzz_trombosit['rendah'], fuzz_leukosit['rendah'])
        rules.append(('R1', alpha_r1, 'tinggi'))

        # R2: Demam ∧ T.Rendah → Tinggi
        alpha_r2 = min(fuzz_suhu['demam'], fuzz_trombosit['rendah'])
        rules.append(('R2', alpha_r2, 'tinggi'))

        # R3: Demam ∧ T.Normal → Sedang
        alpha_r3 = min(fuzz_suhu['demam'], fuzz_trombosit['normal'])
        rules.append(('R3', alpha_r3, 'sedang'))

        # R4: Normal ∧ T.Normal ∧ L.Normal → Rendah
        alpha_r4 = min(fuzz_suhu['normal'], fuzz_trombosit['normal'], fuzz_leukosit['normal'])
        rules.append(('R4', alpha_r4, 'rendah'))

        # R5: Tinggi ∧ L.Normal → Sedang
        alpha_r5 = min(fuzz_suhu['tinggi'], fuzz_leukosit['normal'])
        rules.append(('R5', alpha_r5, 'sedang'))

        return rules

    def komposisi(self, rules):
        """Komposisi dengan operator MAX"""
        risiko = {'rendah': 0, 'sedang': 0, 'tinggi': 0}

        for rule_name, alpha, output in rules:
            risiko[output] = max(risiko[output], alpha)

        return risiko

    def defuzzifikasi_centroid(self, risiko):
        """Defuzzifikasi menggunakan metode centroid"""
        # Area Rendah [0-40]
        if risiko['rendah'] > 0:
            z1 = 20  # centroid area rendah
            luas1 = risiko['rendah'] * 40
            momen1 = risiko['rendah'] * z1 * 40
        else:
            luas1 = 0
            momen1 = 0

        # Area Sedang [30-70]
        if risiko['sedang'] > 0:
            # Area trapesoid terpotong di tinggi risiko['sedang']
            # Untuk penyederhanaan: persegi panjang
            z_start = 30 + (50 - 30) * risiko['sedang']  # 40 untuk μ=0.5
            z_end = 70 - (70 - 50) * risiko['sedang']    # 60 untuk μ=0.5
            lebar = z_end - z_start
            z2 = (z_start + z_end) / 2
            luas2 = risiko['sedang'] * lebar
            momen2 = risiko['sedang'] * z2 * lebar
        else:
            luas2 = 0
            momen2 = 0

        # Area Tinggi [60-100]
        if risiko['tinggi'] > 0:
            z_start = 60 + (80 - 60) * risiko['tinggi']  # 70 untuk μ=0.5
            z_end = 100
            lebar = z_end - z_start
            z3 = (z_start + z_end) / 2
            luas3 = risiko['tinggi'] * lebar
            momen3 = risiko['tinggi'] * z3 * lebar
        else:
            luas3 = 0
            momen3 = 0

        total_luas = luas1 + luas2 + luas3
        total_momen = momen1 + momen2 + momen3

        if total_luas == 0:
            return 0

        centroid = total_momen / total_luas

        print("\n=== DETAIL DEFUZZIFIKASI CENTROID ===")
        print(f"Area Rendah: Luas={luas1:.2f}, Momen={momen1:.2f}")
        print(f"Area Sedang: Luas={luas2:.2f}, Momen={momen2:.2f}")
        print(f"Area Tinggi: Luas={luas3:.2f}, Momen={momen3:.2f}")
        print(f"Total: Luas={total_luas:.2f}, Momen={total_momen:.2f}")
        print(f"Centroid = {total_momen:.2f} / {total_luas:.2f} = {centroid:.2f}")

        return centroid

    def diagnosa(self, suhu, trombosit, leukosit):
        """Proses diagnosa lengkap"""
        print(f"\n📊 DATA PASIEN:")
        print(f"  Suhu: {suhu}°C")
        print(f"  Trombosit: {trombosit} × 10⁹/L")
        print(f"  Leukosit: {leukosit} × 10⁹/L")

        # Fuzzifikasi
        print("\n🔍 LANGKAH 1: FUZZIFIKASI")
        fuzz_suhu = self.fuzzifikasi_suhu(suhu)
        fuzz_trombosit = self.fuzzifikasi_trombosit(trombosit)
        fuzz_leukosit = self.fuzzifikasi_leukosit(leukosit)

        print(f"  Suhu: Normal={fuzz_suhu['normal']:.4f}, Demam={fuzz_suhu['demam']:.4f}, Tinggi={fuzz_suhu['tinggi']:.4f}")
        print(f"  Trombosit: Rendah={fuzz_trombosit['rendah']:.4f}, Normal={fuzz_trombosit['normal']:.4f}")
        print(f"  Leukosit: Rendah={fuzz_leukosit['rendah']:.4f}, Normal={fuzz_leukosit['normal']:.4f}")

        # Inferensi
        print("\n⚙️  LANGKAH 2: INFERENSI (MIN)")
        rules = self.inferensi(fuzz_suhu, fuzz_trombosit, fuzz_leukosit)
        for rule_name, alpha, output in rules:
            print(f"  {rule_name}: α={alpha:.4f} → {output.upper()}")

        # Komposisi
        print("\n🔄 LANGKAH 3: KOMPOSISI (MAX)")
        risiko = self.komposisi(rules)
        for key, value in risiko.items():
            print(f"  Risiko {key.capitalize()}: {value:.4f}")

        # Defuzzifikasi
        print("\n📐 LANGKAH 4: DEFUZZIFIKASI (CENTROID)")
        nilai_risiko = self.defuzzifikasi_centroid(risiko)

        # Interpretasi
        if nilai_risiko < 40:
            kategori = "RENDAH"
            rekomendasi = "Monitor rutin, hidrasi cukup"
        elif nilai_risiko < 70:
            kategori = "SEDANG"
            rekomendasi = "Observasi ketat, cek lab ulang 24 jam"
        else:
            kategori = "TINGGI"
            rekomendasi = "Monitoring intensif, pemeriksaan NS1/IgM, hidrasi adekuat"

        print(f"\n✅ HASIL DIAGNOSA:")
        print(f"  Nilai Risiko DBD: {nilai_risiko:.2f}/100")
        print(f"  Kategori: {kategori}")
        print(f"  Rekomendasi: {rekomendasi}")
        print("="*60 + "\n")

        return nilai_risiko, kategori


# =====================================================
# MAIN PROGRAM - KASUS A
# =====================================================

def main():
    print("LAPORAN UAS - FUZZY LOGIC & ANFIS")
    print("KASUS A: FUZZY MAMDANI")

    # Inisialisasi sistem fuzzy
    mamdani = FuzzyMamdani()

    # Data pasien dari soal
    suhu = 38.5
    trombosit = 120
    leukosit = 3.8

    # Proses diagnosa
    nilai_risiko, kategori = mamdani.diagnosa(suhu, trombosit, leukosit)

    # Kesimpulan
    print("KESIMPULAN")
    print("Metode Fuzzy Mamdani berhasil mendiagnosa risiko DBD")
    print("dengan menggunakan:")
    print("  • Fuzzifikasi: Mengubah input crisp ke fuzzy set")
    print("  • Inferensi MIN: Evaluasi antecedent setiap rule")
    print("  • Komposisi MAX: Gabungkan hasil semua rule")
    print("  • Defuzzifikasi Centroid: Konversi ke nilai crisp")
    print("\nMetode ini cocok untuk sistem diagnosis berbasis")
    print("aturan linguistik dengan interpretabilitas tinggi.")
    print("="*60 + "\n")


if __name__ == "__main__":
    main()

LAPORAN UAS - FUZZY LOGIC & ANFIS
KASUS A: FUZZY MAMDANI

📊 DATA PASIEN:
  Suhu: 38.5°C
  Trombosit: 120 × 10⁹/L
  Leukosit: 3.8 × 10⁹/L

🔍 LANGKAH 1: FUZZIFIKASI
  Suhu: Normal=0.0000, Demam=0.5000, Tinggi=0.5000
  Trombosit: Rendah=0.6000, Normal=0.4000
  Leukosit: Rendah=0.2000, Normal=0.8000

⚙️  LANGKAH 2: INFERENSI (MIN)
  R1: α=0.2000 → TINGGI
  R2: α=0.5000 → TINGGI
  R3: α=0.4000 → SEDANG
  R4: α=0.0000 → RENDAH
  R5: α=0.5000 → SEDANG

🔄 LANGKAH 3: KOMPOSISI (MAX)
  Risiko Rendah: 0.0000
  Risiko Sedang: 0.5000
  Risiko Tinggi: 0.5000

📐 LANGKAH 4: DEFUZZIFIKASI (CENTROID)

=== DETAIL DEFUZZIFIKASI CENTROID ===
Area Rendah: Luas=0.00, Momen=0.00
Area Sedang: Luas=10.00, Momen=500.00
Area Tinggi: Luas=15.00, Momen=1275.00
Total: Luas=25.00, Momen=1775.00
Centroid = 1775.00 / 25.00 = 71.00

✅ HASIL DIAGNOSA:
  Nilai Risiko DBD: 71.00/100
  Kategori: TINGGI
  Rekomendasi: Monitoring intensif, pemeriksaan NS1/IgM, hidrasi adekuat

KESIMPULAN
Metode Fuzzy Mamdani berhasil mendiagn